In [1]:
import pandas as pd

TGIF_TSV = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\TGIF\tgif-v1.0.tsv"   # <-- update if needed
tgif = pd.read_csv(TGIF_TSV, sep="\t", header=None, names=["url", "ref_caption"])

print("Rows:", len(tgif))
print(tgif.head())

Rows: 125782
                                                 url  \
0  https://38.media.tumblr.com/9f6c25cc350f12aa74...   
1  https://38.media.tumblr.com/9ead028ef62004ef6a...   
2  https://38.media.tumblr.com/9f43dc410be85b1159...   
3  https://38.media.tumblr.com/9f659499c8754e40cf...   
4  https://38.media.tumblr.com/9ed1c99afa7d714118...   

                                         ref_caption  
0  a man is glaring, and someone with sunglasses ...  
1           a cat tries to catch a mouse on a tablet  
2                   a man dressed in red is dancing.  
3     an animal comes close to another in the jungle  
4  a man in a hat adjusts his tie and makes a wei...  


In [2]:
import os, hashlib
import requests

CACHE_DIR = "tgif_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def url_to_cache_path(url: str) -> str:
    h = hashlib.md5(url.encode("utf-8")).hexdigest()
    return os.path.join(CACHE_DIR, f"{h}.gif")

def get_gif_bytes(url: str, use_cache=True, timeout=20) -> bytes:
    cache_path = url_to_cache_path(url)
    if use_cache and os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            return f.read()

    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    data = r.content

    if use_cache:
        with open(cache_path, "wb") as f:
            f.write(data)
    return data

In [3]:
import sys

# If main_2.py is in a different folder, add it:

import main_2

# sanity: confirm these exist
needed = ["extract_middle_frame", "detect_content_type", "detect_action",
          "detect_emotion", "detect_objects", "generate_caption"]
for n in needed:
    print(n, "OK" if hasattr(main_2, n) else "MISSING")

 Using device: cpu
 Loading emotion detection model...
 Emotion model loaded!
 Loading object detection model...
 Object detector loaded!
 Loading action detection model...


Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

 Action detector loaded!
extract_middle_frame OK
detect_content_type OK
detect_action OK
detect_emotion OK
detect_objects OK
generate_caption OK


In [4]:
import inspect, importlib, os

print("main_2 imported from:", main_2.__file__)
print("generate_caption source snippet:")
print("\n".join(inspect.getsource(main_2.generate_caption).splitlines()[:25]))

main_2 imported from: d:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\TGIF\main_2.py
generate_caption source snippet:
def generate_caption(
    emotion: str,
    objects: Optional[List[str]] = None,
    action: Optional[str] = None,
    content_type: str = "real_world"
) -> str:
    """
    Emotion-aware TGIF-aligned captioning (emotion ALWAYS included):
    - Always anchors to a PERSON subject for real_world (TGIF-like)
    - Cartoon uses "an animated <adj> person ..."
    - Improved verb inflection (tripling, tying, dancing)
    - Inserts articles in action phrases (punching a bag, grooming a horse)
    - Smarter object prepositions: vehicles -> "near", others -> "with"
    - a/an article selection for object nouns
    - Avoids duplication like "petting cat with a cat"
    """

    objects = objects or []

    # -------------------------
    # 1) Clean / dedupe objects
    # -------------------------
    norm = []
    seen = set()
    for o in objects:


In [5]:
import importlib
import main_2
importlib.reload(main_2)

 Using device: cpu
 Loading emotion detection model...
 Emotion model loaded!
 Loading object detection model...
 Object detector loaded!
 Loading action detection model...


Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

 Action detector loaded!


<module 'main_2' from 'd:\\IIT\\Year 4\\FYP\\Datasets\\GIFGIF_lucas\\TGIF\\main_2.py'>

In [6]:
import io
import numpy as np
from PIL import Image, ImageSequence

def extract_representative_frame_local(gif_bytes: bytes) -> Image.Image | None:
    """
    Simple and reliable: take the middle frame of the GIF.
    """
    try:
        gif = Image.open(io.BytesIO(gif_bytes))
        frames = [fr.convert("RGB") for fr in ImageSequence.Iterator(gif)]
        if len(frames) == 0:
            return None
        return frames[len(frames)//2]
    except Exception as e:
        print(" representative frame error:", e)
        return None
    
def normalize_emotion(emotion_out):
    """
    Convert emotion output into a string label.
    Handles:
      - "happiness"
      - ("happiness", 0.83)
      - {"label":"happiness","score":0.83}
      - None
    """
    if emotion_out is None:
        return "neutral"

    # tuple/list like (label, score)
    if isinstance(emotion_out, (tuple, list)) and len(emotion_out) > 0:
        return str(emotion_out[0])

    # dict like {"label":..., "score":...}
    if isinstance(emotion_out, dict):
        if "label" in emotion_out:
            return str(emotion_out["label"])

    # already a string or something else
    return str(emotion_out)

def detect_objects_wrapper(frame: Image.Image):
    """
    Calls whatever object detection function exists in main_2.
    Returns list[str] (object names) or [].
    """
    # Try common names
    for fn_name in ["detect_objects_in_frame", "detect_objects", "detect_objects_yolo", "get_objects"]:
        if hasattr(main_2, fn_name):
            try:
                out = getattr(main_2, fn_name)(frame)
                if out is None:
                    return []
                return out
            except Exception as e:
                print(f" object fn {fn_name} failed:", e)
                return []
    # If no function found, return empty
    print("️ No object detection function found in main_2.py")
    return []

In [7]:
import io
import numpy as np
from PIL import Image, ImageSequence
from collections import Counter

def extract_k_frames_evenly(gif_bytes: bytes, k: int = 5):
    """
    Extract k evenly spaced RGB frames from a GIF (bytes).
    Returns: list[PIL.Image.Image]
    """
    gif = Image.open(io.BytesIO(gif_bytes))
    frames = [fr.convert("RGB") for fr in ImageSequence.Iterator(gif)]
    if not frames:
        return []
    if k <= 1:
        return [frames[len(frames)//2]]
    idxs = np.linspace(0, len(frames) - 1, k, dtype=int)
    return [frames[i] for i in idxs]

from collections import Counter

def detect_objects_multiframe_vote(
    gif_bytes: bytes,
    k: int = 5,
    top_n: int = 2,
    min_votes: int = 2,
    drop_labels: set | None = None
):
    if drop_labels is None:
        drop_labels = {"person", "man", "woman", "people", "human"}

    frames = extract_k_frames_evenly(gif_bytes, k=k)
    if not frames:
        return []

    all_labels = []
    for fr in frames:
        objs = detect_objects_wrapper(fr) or []
        normed = []
        for o in objs:
            oo = str(o).strip().lower()
            if oo and oo not in drop_labels:
                normed.append(oo)
        # keep top2 per frame
        # raw_fallback: if nothing survives filtering, keep the top-1 raw label (except person-like)
        raw_fallback = []
        for o in objs:
            oo = str(o).strip().lower()
            if oo and oo not in drop_labels:
                raw_fallback.append(oo)
        if not normed and raw_fallback:
            normed = raw_fallback[:1]
        all_labels.extend(normed[:2])

    # If nothing detected across frames, fallback to middle frame
    if not all_labels:
        mid = frames[len(frames)//2]
        fb = [o.lower() for o in (detect_objects_wrapper(mid) or []) if o]
        fb = [o for o in fb if o not in drop_labels]
        return fb[:top_n]

    counts = Counter(all_labels)
    winners = [(lbl, c) for lbl, c in counts.most_common() if c >= min_votes]

    # If voting too strict, fallback to MOST COMMON overall (even if only 1 vote)
    if not winners:
        winners = counts.most_common(top_n)

    return [lbl for lbl, _ in winners[:top_n]]

def clean_objects_for_caption(objs):
    if not objs:
        return []
    # remove very weird common nuisances for TGIF
    block = {"traffic light", "tie", "baseball glove", "tennis racket", "airplane"}
    out = []
    for o in objs:
        oo = o.strip().lower()
        if oo and oo not in block:
            out.append(oo)
    return out[:2]

def insert_article_in_action(action, objects):
    if not action:
        return action

    a = action.lower().strip()
    parts = a.split()
    if len(parts) == 2:
        verb, noun = parts

        # if noun is in detected objects, add article
        obj_set = set([o.lower() for o in (objects or [])])
        if noun in obj_set:
            return f"{verb} a {noun}"

        # animals fallback
        animals = {"dog","cat","horse","bird","bear","elephant","giraffe","zebra","lion","tiger","cow","sheep","monkey","rabbit"}
        if noun in animals:
            return f"{verb} a {noun}"

        # common action objects in TGIF-like captions
        common_nouns = {"bag","ball","pipe","phone","camera","guitar","horse","dog","cat","door","car","bike","bicycle","motorcycle","airplane"}
        if noun in common_nouns:
            return f"{verb} a {noun}"

    return action

def choose_object_preposition(obj: str) -> str:
    vehicles = {"airplane","motorcycle","car","bus","truck","train","boat","ship","bicycle","bike"}
    o = (obj or "").lower().strip()
    return "near" if o in vehicles else "with"

In [8]:
def predict_caption_from_gifbytes(gif_bytes: bytes):
    # 1) representative frame
    frame = extract_representative_frame_local(gif_bytes)
    if frame is None:
        return "", [], None, None   # (caption, objects, action, content_type)

    # 2) content type
    content_type, ct_conf = main_2.detect_content_type(frame)

    # 3) objects (MULTI-FRAME VOTING)
    objects = detect_objects_multiframe_vote(gif_bytes, k=8, top_n=2, min_votes=2)
    objects = clean_objects_for_caption(objects)

    # 4) action (VideoMAE over multiple frames)
    action = None
    if content_type == "real_world":
        action = main_2.detect_action(gif_bytes)

    # 4b) action fallback if VideoMAE returns None
    if action is None:
        # simple motion cue from sampled frames
        frames_for_motion = extract_k_frames_evenly(gif_bytes, k=6)
        if len(frames_for_motion) >= 2:
            arrs = [np.asarray(f.resize((128,128))) for f in frames_for_motion]
            diffs = []
            for a1, a2 in zip(arrs[:-1], arrs[1:]):
                diffs.append(np.mean(np.abs(a2.astype(np.float32)-a1.astype(np.float32))) / 255.0)
            m = float(np.mean(diffs)) if diffs else 0.0
        else:
            m = 0.0
        if m > 0.10:
            action = "moving"
        elif m > 0.05:
            action = "gesturing"
        else:
            action = "reacting"

    # grammar helper
    action = insert_article_in_action(action, objects)

    # 5) emotion
    try:
        emo_out = main_2.detect_emotion(frame)
    except Exception as e:
        print(" detect_emotion failed:", e)
        emo_out = None

    emotion = normalize_emotion(emo_out)

    # 6) caption
    try:
        print(
            "DEBUG objects voted:", objects,
            "| action:", action,
            "| content:", content_type,
            "| emotion:", emotion
        )

        cap = main_2.generate_caption(
            emotion=emotion,
            objects=objects,
            action=action,
            content_type=content_type
        )
        return cap or "", objects, action, content_type

    except Exception as e:
        print(" generate_caption failed:", e)
        return "", objects, action, content_type

In [9]:
row = tgif.iloc[0]
gif_bytes = get_gif_bytes(row["url"])
pred = predict_caption_from_gifbytes(gif_bytes)
print("PRED:", pred)
print("REF :", row["ref_caption"])

 Content Detection - Color: 0.001, Sat: 0.074, Face: False
   → Cartoon (extremely low colors)
DEBUG objects voted: ['cat'] | action: moving | content: cartoon | emotion: negative_subdued
 Generated caption: 'an animated sorrowful person is moving with a cat' (emotion: negative_subdued, objects: ['cat'], action: moving, type: cartoon)
PRED: ('an animated sorrowful person is moving with a cat', ['cat'], 'moving', 'cartoon')
REF : a man is glaring, and someone with sunglasses appears.


In [14]:
import re
import numpy as np
from tqdm import tqdm

# BLEU (NLTK)
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# METEOR (NLTK)
from nltk.translate.meteor_score import meteor_score

# Make sure required NLTK resources exist
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

smooth = SmoothingFunction().method1

def tokenize(s: str):
    s = (s or "").lower().strip()
    # simple tokenization: words + numbers
    return re.findall(r"[a-z0-9']+", s)

# ---------------------------
# ROUGE (pure python)
# ROUGE-1 / ROUGE-2: n-gram overlap F1
# ROUGE-L: LCS-based F1
# ---------------------------

def _ngrams(tokens, n):
    if n <= 0:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def rouge_n_f1(pred_toks, ref_toks, n=1):
    pred_ngr = _ngrams(pred_toks, n)
    ref_ngr  = _ngrams(ref_toks, n)
    if len(pred_ngr) == 0 or len(ref_ngr) == 0:
        return 0.0

    # count overlap with multiplicity
    from collections import Counter
    pc = Counter(pred_ngr)
    rc = Counter(ref_ngr)
    overlap = sum((pc & rc).values())

    prec = overlap / max(1, len(pred_ngr))
    rec  = overlap / max(1, len(ref_ngr))
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

def _lcs_len(a, b):
    # classic DP LCS length (O(len(a)*len(b))) – OK for short captions
    m, n = len(a), len(b)
    if m == 0 or n == 0:
        return 0
    dp = [0] * (n + 1)
    for i in range(1, m + 1):
        prev = 0
        for j in range(1, n + 1):
            tmp = dp[j]
            if a[i-1] == b[j-1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j-1])
            prev = tmp
    return dp[n]

def rouge_l_f1(pred_toks, ref_toks):
    if len(pred_toks) == 0 or len(ref_toks) == 0:
        return 0.0
    lcs = _lcs_len(pred_toks, ref_toks)
    prec = lcs / max(1, len(pred_toks))
    rec  = lcs / max(1, len(ref_toks))
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

def compute_metrics(pred: str, ref: str):
    pred_toks = tokenize(pred)
    ref_toks  = tokenize(ref)

    # BLEU-1..4
    b1 = sentence_bleu([ref_toks], pred_toks, weights=(1,0,0,0), smoothing_function=smooth)
    b2 = sentence_bleu([ref_toks], pred_toks, weights=(0.5,0.5,0,0), smoothing_function=smooth)
    b3 = sentence_bleu([ref_toks], pred_toks, weights=(1/3,1/3,1/3,0), smoothing_function=smooth)
    b4 = sentence_bleu([ref_toks], pred_toks, weights=(0.25,0.25,0.25,0.25), smoothing_function=smooth)

    # METEOR 
    met = meteor_score([ref_toks], pred_toks)

    # ROUGE
    r1 = rouge_n_f1(pred_toks, ref_toks, n=1)
    r2 = rouge_n_f1(pred_toks, ref_toks, n=2)
    rl = rouge_l_f1(pred_toks, ref_toks)

    return b1, b2, b3, b4, met, r1, r2, rl


In [15]:
# =========================
# Semantic Similarity (Cosine) using SentenceTransformer
# =========================
# If sentence-transformers isn't installed, uncomment the pip line.
# !pip install -q sentence-transformers

try:
    from sentence_transformers import SentenceTransformer
except Exception as e:
    raise ImportError(
        "sentence-transformers not installed. Run: pip install sentence-transformers"
    ) from e

import numpy as np

st_model = SentenceTransformer("all-MiniLM-L6-v2")

def cosine_sim_text(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    emb = st_model.encode([a, b], normalize_embeddings=True)
    # cosine of normalized vectors = dot product
    return float(np.dot(emb[0], emb[1]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
empty_obj = 0
empty_act = 0

N = 50
rows = tgif.sample(N, random_state=0).reset_index(drop=True)

bleu1s, bleu2s, bleu3s, bleu4s = [], [], [], []
mets = []
r1s, r2s, rls = [], [], []

coss = []
# ---- Collectors needed for later analysis cells ----
pred_texts = []
ref_texts = []
content_types = []   # "real_world" or "cartoon"

# (optional) also track emotion, objects, actions if you want later analysis
emotions = []
actions = []
objects_list = []

examples = []

for i, row in tqdm(rows.iterrows(), total=len(rows)):
    url = row["url"]
    ref = row["ref_caption"]

    gif_bytes = get_gif_bytes(url)
    if gif_bytes is None:
        continue

    pred, objects, action, content_type = predict_caption_from_gifbytes(gif_bytes)

    if not objects:
        empty_obj += 1
    if not action:
        empty_act += 1

    cos = cosine_sim_text(pred, ref)
    coss.append(cos)

    b1, b2, b3, b4, met, r1, r2, rl = compute_metrics(pred, ref)
    bleu1s.append(b1); bleu2s.append(b2); bleu3s.append(b3); bleu4s.append(b4)
    mets.append(met)
    r1s.append(r1); r2s.append(r2); rls.append(rl)
    pred_texts.append(pred)
    ref_texts.append(ref)
    content_types.append(content_type)

    if i < 8:
        examples.append((pred, ref))

print("Samples scored:", len(bleu1s))
print("Empty objects:", empty_obj, "/", len(bleu1s))
print("Empty actions:", empty_act, "/", len(bleu1s))

print("\n--- N-gram metrics (expected to be low due to emotion injection + single refs) ---")
print("BLEU-1 :", float(np.mean(bleu1s)))
print("BLEU-2 :", float(np.mean(bleu2s)))
print("BLEU-3 :", float(np.mean(bleu3s)))
print("BLEU-4 :", float(np.mean(bleu4s)))
print("METEOR :", float(np.mean(mets)))

print("\n--- Semantic similarity ---")
print(f"COSINE : {np.mean(coss)}")

("--- ROUGE (F1) ---")
print("ROUGE-1:", float(np.mean(r1s)))
print("ROUGE-2:", float(np.mean(r2s)))
print("ROUGE-L:", float(np.mean(rls)))

print("\nExamples:")
for pred, ref in examples:
    print("\nPRED:", pred)
    print("REF :", ref)


print("Collected:", len(pred_texts), len(ref_texts), len(content_types))
print("Content counts:", {k: content_types.count(k) for k in set(content_types)})


  0%|          | 0/50 [00:00<?, ?it/s]

 Content Detection - Color: 0.003, Sat: 0.721, Face: False
   → Cartoon (very low colors + saturated)
DEBUG objects voted: [] | action: moving | content: cartoon | emotion: positive_energetic
 Generated caption: 'an animated delighted person is moving' (emotion: positive_energetic, objects: [], action: moving, type: cartoon)


  2%|▏         | 1/50 [00:02<01:54,  2.34s/it]

 Content Detection - Color: 0.002, Sat: 0.036, Face: True
   → Real-world (face detected)


  4%|▍         | 2/50 [00:07<03:17,  4.12s/it]

...

DEBUG objects voted: ['cat', 'bed'] | action: moving | content: cartoon | emotion: positive_energetic
 Generated caption: 'an animated delighted person is moving with a cat' (emotion: positive_energetic, objects: ['cat', 'bed'], action: moving, type: cartoon)
 Content Detection - Color: 0.001, Sat: 0.000, Face: True
   → Real-world (face detected)


 98%|█████████▊| 49/50 [02:14<00:02,  2.70s/it]

DEBUG objects voted: [] | action: moving | content: real_world | emotion: positive_calm
 Generated caption: 'a soothing person is moving' (emotion: positive_calm, objects: [], action: moving, type: real_world)
 Content Detection - Color: 0.000, Sat: 0.000, Face: True
   → Real-world (face detected)


100%|██████████| 50/50 [02:18<00:00,  2.76s/it]

DEBUG objects voted: ['cat'] | action: petting a cat | content: real_world | emotion: positive_energetic
 Generated caption: 'an energetic person is petting a cat' (emotion: positive_energetic, objects: ['cat'], action: petting a cat, type: real_world)
Samples scored: 50
Empty objects: 28 / 50
Empty actions: 0 / 50

--- N-gram metrics (expected to be low due to emotion injection + single refs) ---
BLEU-1 : 0.14138399083206699
BLEU-2 : 0.049636890986097144
BLEU-3 : 0.027402539704477643
BLEU-4 : 0.0220786713279467
METEOR : 0.09898136582718103

--- Semantic similarity ---
COSINE : 0.2712600123882294
ROUGE-1: 0.19321976638847013
ROUGE-2: 0.017504017551076376
ROUGE-L: 0.1769584431457227

Examples:

PRED: an animated delighted person is moving
REF : a female athlete is performing the long jump.

PRED: a sad person is moving
REF : a man is dancing and moving in front of the cameras

PRED: an animated terrified person is gesturing with an apple
REF : a woman is lighting and smoking from a pipe

In [24]:
print(len(r1s), len(r2s), len(rls), len(coss))

50 50 50 50


## Additional Evaluation Extensions (Real-world focus + extra metrics)

This section adds:
- **Real-world focus evaluation** (filtering out animated/cartoon samples)
- **Token-level Precision / Recall / F1** (bag-of-words overlap)
- **BERTScore** (semantic metric robust to paraphrasing)
- **Emotion vs No-emotion ablation** (quantifies lexical penalty of emotion injection)

> **System design note (for thesis / future enhancement):**
> The current pipeline is optimized for **real-world GIFs**. If an input is detected as animated/cartoon, the frontend can show a **warning** (e.g., “Animated GIF detected — results may be less accurate; real-world GIFs recommended.”). Supporting animated GIFs can be presented as future work (domain-adapted action/object models).


In [29]:
import numpy as np

# --- Real-world focus evaluation ---
# We prefer splitting/evaluating on the 'content' label produced by your pipeline.
# Expected list name from earlier cells: content_types (or clip_types).
# If not found, this cell will skip.

def _get_content_list():
    for name in ["content_types", "clip_types", "types", "content_list"]:
        if name in globals() and isinstance(globals()[name], list) and len(globals()[name]) > 0:
            return globals()[name], name
    return None, None

content_list, content_var = _get_content_list()

if content_list is None:
    print("️ Could not find a content-type list (e.g., content_types). Skipping real-world-only split.")
else:
    # normalize labels
    norm = []
    for x in content_list:
        s = str(x).strip().lower()
        if s in ["real", "realworld", "real_world", "real-world", "photo", "photographic"]:
            norm.append("real_world")
        elif s in ["cartoon", "animated", "animation", "anime", "toon"]:
            norm.append("cartoon")
        else:
            norm.append(s)
    content_list = norm

    real_idxs = [i for i,t in enumerate(content_list) if t == "real_world"]
    cart_idxs = [i for i,t in enumerate(content_list) if t == "cartoon"]
    other_idxs = [i for i,t in enumerate(content_list) if t not in ["real_world","cartoon"]]

    print(f"Using content labels from: {content_var}")
    print(f"Real-world samples : {len(real_idxs)} / {len(content_list)}")
    print(f"Animated/cartoon   : {len(cart_idxs)} / {len(content_list)}")
    if len(other_idxs)>0:
        print(f"Other/unknown      : {len(other_idxs)} / {len(content_list)}")

    import numpy as np

    real_idxs = [i for i,c in enumerate(content_types) if c == "real_world"]
    cart_idxs = [i for i,c in enumerate(content_types) if c == "cartoon"]

    import numpy as np

    def _mean(arr, idxs):
        if len(idxs) == 0:
            return None
        arr = np.asarray(arr, dtype=float)
        idxs = np.asarray(idxs, dtype=int)
        return float(arr[idxs].mean())

    print("\n=== Split evaluation (heuristic content type) ===")
    print("real_world n =", len(real_idxs))
    print("cartoon    n =", len(cart_idxs))

    print("real idx sample:", real_idxs[:5])
    print("cart idx sample:", cart_idxs[:5])
    print("Example r1 real:", r1s[real_idxs[0]], "cart:", r1s[cart_idxs[0]])
    print("Mean r1 global:", float(np.mean(r1s)))
    print("Mean r1 real  :", float(np.mean(np.asarray(r1s)[np.asarray(real_idxs, dtype=int)])))
    print("Mean r1 cart  :", float(np.mean(np.asarray(r1s)[np.asarray(cart_idxs, dtype=int)])))

    print("\n[REAL-WORLD]")
    print("BLEU-1 :", _mean(bleu1s, real_idxs))
    print("BLEU-2 :", _mean(bleu2s, real_idxs))
    print("BLEU-3 :", _mean(bleu3s, real_idxs))
    print("BLEU-4 :", _mean(bleu4s, real_idxs))
    print("METEOR :", _mean(mets,  real_idxs))
    print("ROUGE-1:", _mean(r1s, real_idxs))
    print("ROUGE-2:", _mean(r2s, real_idxs))
    print("ROUGE-L:", _mean(rls, real_idxs))
    print("COSINE :", _mean(coss, real_idxs))

    print("\n[CARTOON]")
    print("BLEU-1 :", _mean(bleu1s, cart_idxs))
    print("BLEU-2 :", _mean(bleu2s, cart_idxs))
    print("BLEU-3 :", _mean(bleu3s, cart_idxs))
    print("BLEU-4 :", _mean(bleu4s, cart_idxs))
    print("METEOR :", _mean(mets,  cart_idxs))
    print("ROUGE-1:", _mean(r1s, cart_idxs))
    print("ROUGE-2:", _mean(r2s, cart_idxs))
    print("ROUGE-L:", _mean(rls, cart_idxs))
    print("COSINE :", _mean(coss, cart_idxs))


Using content labels from: content_types
Real-world samples : 23 / 50
Animated/cartoon   : 27 / 50

=== Split evaluation (heuristic content type) ===
real_world n = 23
cartoon    n = 27
real idx sample: [1, 3, 4, 5, 7]
cart idx sample: [0, 2, 6, 10, 11]
Example r1 real: 0.37499999999999994 cart: 0.14285714285714288
Mean r1 global: 0.19321976638847013
Mean r1 real  : 0.23686687529129308
Mean r1 cart  : 0.15603889584162087

[REAL-WORLD]
BLEU-1 : 0.14429034951862915
BLEU-2 : 0.05329568131734268
BLEU-3 : 0.028460493259694776
BLEU-4 : 0.022935955031963773
METEOR : 0.11305754555381642
ROUGE-1: 0.23686687529129308
ROUGE-2: 0.02386946386946387
ROUGE-L: 0.2285311738609625
COSINE : 0.3202868210880653

[CARTOON]
BLEU-1 : 0.13890820380277333
BLEU-2 : 0.046520143666887975
BLEU-3 : 0.02650132000929269
BLEU-4 : 0.021348392617117334
METEOR : 0.08699054606004716
ROUGE-1: 0.15603889584162087
ROUGE-2: 0.012081600316894434
ROUGE-L: 0.13302611698088881
COSINE : 0.22949643460688768


In [30]:
# Aliases so later analysis cells can use standard names
rouge1s, rouge2s, rougels = r1s, r2s, rls
cos_sims = coss

## Token-level Precision / Recall / F1 (bag-of-words overlap)

This is a simple lexical overlap measure that is easier to interpret than BLEU, and complements ROUGE.


In [31]:
import re
from collections import Counter
import numpy as np

def simple_tokenize(s: str):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    toks = [t for t in s.split() if t.strip()]
    return toks

def token_prf(pred: str, ref: str):
    pt = simple_tokenize(pred)
    rt = simple_tokenize(ref)
    if len(pt)==0 and len(rt)==0:
        return 1.0, 1.0, 1.0
    if len(pt)==0 or len(rt)==0:
        return 0.0, 0.0, 0.0

    pc = Counter(pt)
    rc = Counter(rt)
    overlap = sum((pc & rc).values())

    precision = overlap / max(1, sum(pc.values()))
    recall = overlap / max(1, sum(rc.values()))
    f1 = 0.0 if (precision+recall)==0 else (2*precision*recall/(precision+recall))
    return precision, recall, f1

# expects pred_texts and ref_texts lists
if "pred_texts" not in globals() or "ref_texts" not in globals():
    print("️ pred_texts/ref_texts not found. Make sure you collect them in the main loop.")
else:
    tok_ps, tok_rs, tok_f1s = [], [], []
    for p, r in zip(pred_texts, ref_texts):
        pr, rc, f1 = token_prf(p, r)
        tok_ps.append(pr); tok_rs.append(rc); tok_f1s.append(f1)

    print("\n--- Token overlap (bag-of-words) ---")
    print("Token Precision:", float(np.mean(tok_ps)))
    print("Token Recall   :", float(np.mean(tok_rs)))
    print("Token F1       :", float(np.mean(tok_f1s)))

    # Optional: real-world-only token PRF if content labels exist
    content_list, _ = _get_content_list() if '_get_content_list' in globals() else (None, None)
    if content_list is not None:
        norm = []
        for x in content_list:
            s = str(x).strip().lower()
            if s in ["real", "realworld", "real_world", "real-world"]:
                norm.append("real_world")
            elif s in ["cartoon", "animated", "animation", "anime", "toon"]:
                norm.append("cartoon")
            else:
                norm.append(s)
        real_idxs = [i for i,t in enumerate(norm) if t=="real_world"]
        if len(real_idxs)>0:
            print("\nToken F1 (real-world only):", float(np.mean(np.array(tok_f1s)[real_idxs])))



--- Token overlap (bag-of-words) ---
Token Precision: 0.24993428793428796
Token Recall   : 0.1667673878409173
Token F1       : 0.19295832847997338

Token F1 (real-world only): 0.23629853201195236


## BERTScore (semantic similarity robust to paraphrasing)

If not installed yet, run: `!pip install -q bert-score`


In [26]:
%pip install -q bert-score

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
# If needed:
# !pip install -q bert-score

import numpy as np

if "pred_texts" not in globals() or "ref_texts" not in globals():
    print("️ pred_texts/ref_texts not found. Make sure you collect them in the main loop.")
else:
    try:
        from bert_score import score as bertscore
        P, R, F1 = bertscore(pred_texts, ref_texts, lang="en", model_type="roberta-base", verbose=False)
        print("\n--- BERTScore (roberta-base) ---")
        print("BERTScore Precision:", float(P.mean()))
        print("BERTScore Recall   :", float(R.mean()))
        print("BERTScore F1       :", float(F1.mean()))

        # Optional: real-world-only
        content_list, _ = _get_content_list() if '_get_content_list' in globals() else (None, None)
        if content_list is not None:
            norm = []
            for x in content_list:
                s = str(x).strip().lower()
                if s in ["real", "realworld", "real_world", "real-world"]:
                    norm.append("real_world")
                elif s in ["cartoon", "animated", "animation", "anime", "toon"]:
                    norm.append("cartoon")
                else:
                    norm.append(s)
            real_idxs = [i for i,t in enumerate(norm) if t=="real_world"]
            if len(real_idxs)>0:
                print("BERTScore F1 (real-world only):", float(F1[real_idxs].mean()))
    except Exception as e:
        print("️ BERTScore failed. Error:", repr(e))
        print("If bert-score isn't installed, run: !pip install -q bert-score")


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

d:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\gifgif-env\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\akind\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- BERTScore (roberta-base) ---
BERTScore Precision: 0.8858471512794495
BERTScore Recall   : 0.8773565888404846
BERTScore F1       : 0.8814504146575928
BERTScore F1 (real-world only): 0.8836421370506287


## Emotion vs No-emotion Ablation

We remove the emotion adjective from the generated caption and recompute metrics.
This quantifies how much **emotion injection reduces lexical overlap** with neutral TGIF references.


In [33]:
import re
import numpy as np

def strip_emotion_phrase(caption: str):
    c = caption.strip()
    # a/an + optional animated + adjective + person is -> a/an + optional animated + person is
    c2 = re.sub(r"^(a|an)\s+(animated\s+)?([a-z]+)\s+person(\s+is\s+)", r"\1 \2person\4", c, flags=re.IGNORECASE)
    c2 = re.sub(r"\s{2,}", " ", c2).strip()
    return c2

if "pred_texts" not in globals() or "ref_texts" not in globals():
    print("️ pred_texts/ref_texts not found. Make sure you collect them in the main loop.")
elif "compute_metrics" not in globals():
    print("️ compute_metrics() not found. Ensure your metric function cell ran.")
else:
    pred_noemo = [strip_emotion_phrase(p) for p in pred_texts]

    bleu4_ne, met_ne, rl_ne = [], [], []
    for p, r in zip(pred_noemo, ref_texts):
        b1, b2, b3, b4, met, rr1, rr2, rrl = compute_metrics(p, r)
        bleu4_ne.append(b4); met_ne.append(met); rl_ne.append(rrl)

    # cosine similarity for stripped captions if cosine function exists
    cos_ne = None
    if "cosine_sim_sentence" in globals():
        cos_ne = [cosine_sim_sentence(p, r) for p, r in zip(pred_noemo, ref_texts)]

    print("\n=== Ablation: WITH emotion (original) ===")
    if "bleu4s" in globals(): print("BLEU-4 :", float(np.mean(bleu4s)))
    if "mets" in globals():  print("METEOR :", float(np.mean(mets)))
    if "rougels" in globals(): print("ROUGE-L:", float(np.mean(rougels)))
    if "cos_sims" in globals(): print("COSINE :", float(np.mean(cos_sims)))

    print("\n=== Ablation: NO emotion (stripped adjective) ===")
    print("BLEU-4 :", float(np.mean(bleu4_ne)))
    print("METEOR :", float(np.mean(met_ne)))
    print("ROUGE-L:", float(np.mean(rl_ne)))
    if cos_ne is not None:
        print("COSINE :", float(np.mean(cos_ne)))

    # Optional: show a couple examples
    print("\nExamples (original -> stripped):")
    for i in range(min(3, len(pred_texts))):
        print(f"- {pred_texts[i]}  ->  {pred_noemo[i]}")



=== Ablation: WITH emotion (original) ===
BLEU-4 : 0.0220786713279467
METEOR : 0.09898136582718103
ROUGE-L: 0.1769584431457227
COSINE : 0.2712600123882294

=== Ablation: NO emotion (stripped adjective) ===
BLEU-4 : 0.021651853899796875
METEOR : 0.09998238556411558
ROUGE-L: 0.18748467612762945

Examples (original -> stripped):
- an animated delighted person is moving  ->  an animated person is moving
- a sad person is moving  ->  a person is moving
- an animated terrified person is gesturing with an apple  ->  an animated person is gesturing with an apple
